In [11]:
import math
import sys
from functools import partial
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torchvision.transforms as T
from torch.cuda.amp import GradScaler, autocast
from torch.utils.data import DataLoader, WeightedRandomSampler
from transformers import get_cosine_schedule_with_warmup
import timm
from tqdm.auto import tqdm
from sklearn.metrics import classification_report, confusion_matrix

sys.path.append(".")

# CropR modules (devono essere nella working dir o su sys.path)
from cropr import Cropr
from src.vision_transformer_copr import VisionTransformer as CroprVisionTransformer

from src.dataset import HistologicalImageDataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"PyTorch  : {torch.__version__}")
print(f"timm     : {timm.__version__}")

Device: cuda
PyTorch  : 2.6.0+cu124
timm     : 0.9.16


In [12]:
CFG = dict(
    # ---- Dati ----
    data_dir        = "/data/BREAKHIS",
    dataset_name  = "BREAKHIS",  
    img_size        = 224,
    batch_size      = 8,
    num_workers     = 4,

    # ---- Backbone UNI ----
    patch_size      = 16,
    embed_dim       = 1024,          # ViT-L
    depth           = 24,
    num_heads       = 16,
    mlp_ratio       = 4.0,
    init_values     = 1e-5,          # layer-scale UNI
    drop_path       = 0.2,
    global_pool     = "token",       # CLS token (come UNI originale)

    # ---- CropR ----
    use_cropr           = True,
    cropr_pruning_rate  = 8,         # token rimossi per blocco
    cropr_llf           = False,     # Last Layer Fusion
    cropr_num_queries   = 1,
    cropr_num_heads     = 1,
    cropr_pre_attn_norm = False,
    cropr_q_proj        = False,
    cropr_k_proj        = False,
    cropr_v_proj        = False,
    cropr_mlp           = True,
    cropr_mlp_ratio     = 4.0,

    # ---- Ottimizzazione ----
    lr              = 1e-3,          # LR base (scalato linearmente con batch)
    min_lr          = 1e-6,
    layer_decay     = 0.75,          # LLRD — layer-wise LR decay
    weight_decay    = 0.05,
    epochs          = 20,
    warmup_epochs   = 2,
    accum_steps     = 1,             # gradient accumulation
    max_norm        = 1.0,           # gradient clipping
    label_smoothing = 0.1,

    # ---- Misc ----
    seed            = 42,
)

CFG["dataset_name"] = Path(CFG["data_dir"]).name
CFG["output_dir"]   = Path(f"checkpoints/{CFG['dataset_name']}/uni_cropr")
CFG["output_dir"].mkdir(parents=True, exist_ok=True)


torch.manual_seed(CFG["seed"])
np.random.seed(CFG["seed"])
print(f"Output dir: {CFG['output_dir']}")

# %% [markdown]
# ## 2 · Dati & Augmentation

# %%
train_tf = T.Compose([
    T.RandomHorizontalFlip(),
    T.RandomVerticalFlip(),
    T.RandomApply([T.RandomRotation((90, 90))], p=0.5),
    T.RandomApply([T.ColorJitter(0.2, 0.2, 0.1, 0.05)], p=0.5),
    T.Resize((CFG["img_size"], CFG["img_size"])),
    T.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
])

eval_tf = T.Compose([
    T.Resize((CFG["img_size"], CFG["img_size"])),
    T.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
])

train_ds = HistologicalImageDataset(f"{CFG['data_dir']}/train", transform=train_tf)
val_ds   = HistologicalImageDataset(f"{CFG['data_dir']}/val",   transform=eval_tf)
test_ds  = HistologicalImageDataset(f"{CFG['data_dir']}/test",  transform=eval_tf)

# Sampler bilanciato per gestire lo sbilanciamento delle classi
counts  = np.bincount(train_ds.labels)
weights = torch.from_numpy((1.0 / counts)[train_ds.labels]).double()
sampler = WeightedRandomSampler(weights, len(weights), replacement=True)

kw = dict(batch_size=CFG["batch_size"], num_workers=CFG["num_workers"],
          pin_memory=True, persistent_workers=True)

train_loader = DataLoader(train_ds, sampler=sampler, drop_last=True, **kw)
val_loader   = DataLoader(val_ds,  shuffle=False, **kw)
test_loader  = DataLoader(test_ds, shuffle=False, **kw)

CLASS_NAMES = train_ds.class_names
N_CLASSES   = len(CLASS_NAMES)
print(f"Classi ({N_CLASSES}): {CLASS_NAMES}")
print(f"Train: {len(train_ds)}  |  Val: {len(val_ds)}  |  Test: {len(test_ds)}")

Output dir: checkpoints/BREAKHIS/uni_cropr
Loading from /data/BREAKHIS/train...


Loading dataset from disk:   0%|          | 0/29 [00:00<?, ?it/s]

Loaded 25880 samples, 8 classes
Class distribution:
  adenosis: 1139 (4.4%)
  fibroadenoma: 3685 (14.2%)
  phyllodes_tumor: 1419 (5.5%)
  tubular_adenoma: 1642 (6.3%)
  ductal_carcinoma: 11717 (45.3%)
  lobular_carcinoma: 1927 (7.4%)
  mucinous_carcinoma: 2446 (9.5%)
  papillary_carcinoma: 1905 (7.4%)
Loading from /data/BREAKHIS/val...
Loaded 6832 samples, 8 classes
Class distribution:
  adenosis: 541 (7.9%)
  fibroadenoma: 692 (10.1%)
  phyllodes_tumor: 423 (6.2%)
  tubular_adenoma: 601 (8.8%)
  ductal_carcinoma: 2769 (40.5%)
  lobular_carcinoma: 601 (8.8%)
  mucinous_carcinoma: 757 (11.1%)
  papillary_carcinoma: 448 (6.6%)
Loading from /data/BREAKHIS/test...
Loaded 6833 samples, 8 classes
Class distribution:
  adenosis: 540 (7.9%)
  fibroadenoma: 693 (10.1%)
  phyllodes_tumor: 423 (6.2%)
  tubular_adenoma: 602 (8.8%)
  ductal_carcinoma: 2769 (40.5%)
  lobular_carcinoma: 602 (8.8%)
  mucinous_carcinoma: 757 (11.1%)
  papillary_carcinoma: 447 (6.5%)
Classi (8): ['adenosis', 'fibroadeno

In [14]:
import wandb
DATASET_NAME = CFG["dataset_name"]
wandb.init(
    project="uni-cropr",
    name=f"{CFG['dataset_name']}_cropr_pr{CFG['cropr_pruning_rate']}",
    config=CFG,
    tags=[DATASET_NAME ],
)

wandb: Currently logged in as: vincenzo-civale (vincenzo-civale-universi-degli-studi-di-firenze) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [15]:
class UNICroprVisionTransformer(CroprVisionTransformer):
    """
    VisionTransformer con CropR adattato per UNI:
    - i blocchi vengono ri-creati con `init_values` (layer-scale) per
      permettere il caricamento corretto dei pesi pre-addestrati di UNI.
    """

    def __init__(self, cropr_cfg, init_values: float = 1e-5, **kwargs):
        super().__init__(cropr_cfg, **kwargs)

        # Ri-crea i blocchi con layer-scale per allinearsi ad UNI
        dpr = [
            x.item()
            for x in torch.linspace(0, kwargs["drop_path_rate"], len(self.blocks))
        ]
        # Ultimo blocco senza drop-path quando CropR è attivo
        if cropr_cfg["use_cropr"]:
            dpr[-1] = 0.0

        for blk_idx in range(len(self.blocks)):
            self.blocks[blk_idx] = timm.models.vision_transformer.Block(
                dim=self.embed_dim,
                num_heads=kwargs["num_heads"],
                qkv_bias=True,
                init_values=init_values,          # <— layer-scale UNI
                drop_path=dpr[blk_idx],
                norm_layer=partial(nn.LayerNorm, eps=1e-6),
            )


def build_model(n_classes: int, cfg: dict) -> nn.Module:
    """
    Costruisce UNICroprVisionTransformer e carica i pesi pre-addestrati di UNI.
    I pesi dei moduli CropR e della nuova head di classificazione vengono
    inizializzati casualmente (strict=False).
    """
    cropr_cfg = {
        "use_cropr"     : cfg["use_cropr"],
        "pruning_rate"  : cfg["cropr_pruning_rate"],
        "llf"           : cfg["cropr_llf"],
        "num_queries"   : cfg["cropr_num_queries"],
        "num_heads"     : cfg["cropr_num_heads"],
        "pre_attn_norm" : cfg["cropr_pre_attn_norm"],
        "q_proj"        : cfg["cropr_q_proj"],
        "k_proj"        : cfg["cropr_k_proj"],
        "v_proj"        : cfg["cropr_v_proj"],
        "mlp"           : cfg["cropr_mlp"],
        "mlp_ratio"     : cfg["cropr_mlp_ratio"],
        "training"      : True,
    }

    model_kwargs = dict(
        num_classes     = n_classes,
        img_size        = cfg["img_size"],
        patch_size      = cfg["patch_size"],
        embed_dim       = cfg["embed_dim"],
        depth           = cfg["depth"],
        num_heads       = cfg["num_heads"],
        mlp_ratio       = cfg["mlp_ratio"],
        drop_path_rate  = cfg["drop_path"],
        global_pool     = cfg["global_pool"],
        class_token     = True,
    )

    model = UNICroprVisionTransformer(
        cropr_cfg,
        init_values=cfg["init_values"],
        **model_kwargs,
    )

    # --- Caricamento pesi UNI ---
    print("Caricamento pesi pre-addestrati UNI...")
    uni_backbone = timm.create_model(
        "hf-hub:MahmoodLab/uni",
        pretrained=True,
        init_values=cfg["init_values"],
        dynamic_img_size=True,
        num_classes=0,           # senza head di classificazione
    )

    state_dict = uni_backbone.state_dict()
    msg = model.load_state_dict(state_dict, strict=False)
    print(f"  Chiavi mancanti (nuovi moduli): {len(msg.missing_keys)}")
    print(f"  Chiavi inattese (rimosse):      {len(msg.unexpected_keys)}")
    del uni_backbone

    total   = sum(p.numel() for p in model.parameters()) / 1e6
    trainbl = sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6
    print(f"Parametri totali:    {total:.1f}M")
    print(f"Parametri trainable: {trainbl:.1f}M")

    return model


model = build_model(N_CLASSES, CFG).to(device, dtype=torch.bfloat16)
# Abilita gradient checkpointing sul backbone
model.set_grad_checkpointing(enable=True)

Cropr schedule (tokens remaining): [188, 180, 172, 164, 156, 148, 140, 132, 124, 116, 108, 100, 92, 84, 76, 68, 60, 52, 44, 36, 28, 20, 12]
Caricamento pesi pre-addestrati UNI...
  Chiavi mancanti (nuovi moduli): 255
  Chiavi inattese (rimosse):      0
Parametri totali:    496.7M
Parametri trainable: 496.7M


In [16]:
def param_groups_lrd(
    model: nn.Module,
    weight_decay: float = 0.05,
    no_weight_decay_list: set = frozenset(),
    layer_decay: float = 0.75,
) -> list:
    """
    Assegna a ogni gruppo di parametri un learning rate scalato per layer.
    - Layer 0  : patch_embed, cls_token, pos_embed  → lr × decay^(depth+1)
    - Layer i  : blocks[i-1]                        → lr × decay^(depth+1-i)
    - Layer D+1: norm, head, cropr                  → lr × 1.0
    """
    param_group_names: dict = {}
    param_groups: dict      = {}

    # Conta quanti "strati" ha il backbone
    num_layers = len(model.blocks) + 1   # blocks + stem

    def get_layer_id(name: str) -> int:
        if name in ("cls_token", "pos_embed"):
            return 0
        if name.startswith("patch_embed"):
            return 0
        if name.startswith("blocks."):
            layer_idx = int(name.split(".")[1])
            return layer_idx + 1       # 1 … depth
        # norm, head, cropr → ultimo gruppo (no decay)
        return num_layers

    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue

        # Nessun weight decay su bias e LayerNorm
        if param.ndim == 1 or name.endswith(".bias") or name in no_weight_decay_list:
            wd = 0.0
            g_decay = "no_decay"
        else:
            wd = weight_decay
            g_decay = "decay"

        layer_id  = get_layer_id(name)
        lr_scale  = layer_decay ** (num_layers - layer_id)
        group_key = f"layer_{layer_id:02d}_{g_decay}"

        if group_key not in param_group_names:
            param_group_names[group_key] = {
                "lr_scale"     : lr_scale,
                "weight_decay" : wd,
                "params"       : [],
            }
            param_groups[group_key] = {
                "lr_scale"     : lr_scale,
                "weight_decay" : wd,
                "params"       : [],
            }

        param_group_names[group_key]["params"].append(name)
        param_groups[group_key]["params"].append(param)

    # Stampa un riassunto compatto
    print(f"{'Gruppo':<30} {'lr_scale':>10}  {'#params':>8}")
    print("-" * 54)
    for k, v in sorted(param_group_names.items()):
        n_p = len(v["params"])
        print(f"  {k:<28} {v['lr_scale']:>10.4f}  {n_p:>8}")

    return list(param_groups.values())


# LR effettivo = lr × (batch_size / 256)  — regola di scaling lineare
effective_lr = CFG["lr"] * CFG["batch_size"] * CFG["accum_steps"] / 256.0
print(f"\nLR effettivo (scaling lineare): {effective_lr:.2e}")

param_groups = param_groups_lrd(
    model,
    weight_decay=CFG["weight_decay"],
    no_weight_decay_list=model.no_weight_decay(),
    layer_decay=CFG["layer_decay"],
)

optimizer = torch.optim.AdamW(param_groups, lr=effective_lr, weight_decay=CFG["weight_decay"])



LR effettivo (scaling lineare): 3.13e-05
Gruppo                           lr_scale   #params
------------------------------------------------------
  layer_00_decay                   0.0008         1
  layer_00_no_decay                0.0008         3
  layer_01_decay                   0.0010         4
  layer_01_no_decay                0.0010        10
  layer_02_decay                   0.0013         4
  layer_02_no_decay                0.0013        10
  layer_03_decay                   0.0018         4
  layer_03_no_decay                0.0018        10
  layer_04_decay                   0.0024         4
  layer_04_no_decay                0.0024        10
  layer_05_decay                   0.0032         4
  layer_05_no_decay                0.0032        10
  layer_06_decay                   0.0042         4
  layer_06_no_decay                0.0042        10
  layer_07_decay                   0.0056         4
  layer_07_no_decay                0.0056        10
  layer_08_decay   

In [17]:
criterion = nn.CrossEntropyLoss(label_smoothing=CFG["label_smoothing"])

# Scheduler cosine con warmup (uno step per batch, come nel codice CropR)
steps_per_epoch = len(train_loader) // CFG["accum_steps"]
total_steps     = CFG["epochs"] * steps_per_epoch
warmup_steps    = CFG["warmup_epochs"] * steps_per_epoch

# Costruiamo manualmente il cosine schedule con warmup
def cosine_schedule_with_warmup(
    base_lr: float, min_lr: float,
    total_steps: int, warmup_steps: int,
) -> np.ndarray:
    """Ritorna un array di valori LR per ogni step di gradient update."""
    schedule = []
    for t in range(total_steps):
        if t < warmup_steps:
            lr = base_lr * t / max(1, warmup_steps)
        else:
            progress = (t - warmup_steps) / max(1, total_steps - warmup_steps)
            lr = min_lr + 0.5 * (base_lr - min_lr) * (1 + math.cos(math.pi * progress))
        schedule.append(lr)
    return np.array(schedule)


lr_schedule = cosine_schedule_with_warmup(
    effective_lr, CFG["min_lr"], total_steps, warmup_steps
)

# Mixed Precision
scaler = GradScaler()

print(f"Steps totali: {total_steps}  |  Warmup steps: {warmup_steps}")

Steps totali: 64700  |  Warmup steps: 6470


/tmp/ipykernel_869172/3506555803.py:30: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


In [ ]:
def run_epoch(
    model,
    loader,
    criterion,
    optimizer=None,
    scheduler=None,
    grad_offset: int = 0,
):
    """
    Esegue un'epoca di training o valutazione.

    Args:
        grad_offset: indice iniziale nel lr_schedule (usato solo in training).

    Returns:
        avg_loss, accuracy, grad_step_finale
    """
    training = optimizer is not None
    model.train() if training else model.eval()

    total_loss, correct, total = 0.0, 0, 0
    grad_i = grad_offset

    with torch.set_grad_enabled(training):
        for batch_i, (imgs, labels) in enumerate(tqdm(loader, leave=False)):
            imgs   = imgs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            # Aggiorna LR prima del forward (ogni accum_steps)
            if training and batch_i % CFG["accum_steps"] == 0:
                current_lr = lr_schedule[min(grad_i, len(lr_schedule) - 1)]
                for pg in optimizer.param_groups:
                    pg["lr"] = current_lr * pg.get("lr_scale", 1.0)

            with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
                output = model(imgs)

                # Multi-output CropR (training) vs single tensor (eval)
                if isinstance(output, torch.Tensor):
                    loss = criterion(output, labels)
                    logits_main = output
                else:
                    losses = {
                        ("main" if i == 0 else f"aux_{i}"): criterion(out, labels)
                        for i, out in enumerate(output)
                    }
                    loss = sum(losses.values())
                    logits_main = output[0]

            if training:
                (loss / CFG["accum_steps"]).backward()
                if (batch_i + 1) % CFG["accum_steps"] == 0:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), CFG["max_norm"])
                    optimizer.step()
                    optimizer.zero_grad()
                    grad_i += 1

            total_loss += loss.item() * len(labels)
            correct    += (logits_main.argmax(1) == labels).sum().item()
            total      += len(labels)

    return total_loss / total, correct / total, grad_i


# %% [markdown]
# ## 7 · Training

# %%
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
best_val_acc, best_epoch = 0.0, 0
grad_step = 0   # puntatore al lr_schedule

for epoch in range(1, CFG["epochs"] + 1):

    tr_loss, tr_acc, grad_step = run_epoch(
        model, train_loader, criterion,
        optimizer=optimizer, scheduler=lr_schedule,
        grad_offset=grad_step,
    )
    vl_loss, vl_acc, _ = run_epoch(model, val_loader, criterion)

    history["train_loss"].append(tr_loss)
    history["train_acc"].append(tr_acc)
    history["val_loss"].append(vl_loss)
    history["val_acc"].append(vl_acc)

    current_lr = optimizer.param_groups[-1]["lr"]
    print(
        f"Epoch {epoch:02d}/{CFG['epochs']}  "
        f"train_loss={tr_loss:.4f}  train_acc={tr_acc:.4f}  "
        f"val_loss={vl_loss:.4f}  val_acc={vl_acc:.4f}  "
        f"lr={current_lr:.2e}"
    )

    wandb.log({
    "epoch"       : epoch,
    "train/loss"  : tr_loss,
    "train/acc"   : tr_acc,
    "val/loss"    : vl_loss,
    "val/acc"     : vl_acc,
    "lr"          : optimizer.param_groups[-1]["lr"],
    })

    if vl_acc > best_val_acc:
        best_val_acc, best_epoch = vl_acc, epoch
        torch.save(model.state_dict(), CFG["output_dir"] / "best_model.pt")
        wandb.summary["best_val_acc"]   = best_val_acc
        wandb.summary["best_epoch"]     = best_epoch
        print(f"  ✅ Salvato best model (val_acc={best_val_acc:.4f})")



print(f"\nBest val_acc={best_val_acc:.4f} @ epoch {best_epoch}")

  0%|          | 0/3235 [00:00<?, ?it/s]

  0%|          | 0/854 [00:00<?, ?it/s]

Epoch 01/20  train_loss=49.0118  train_acc=0.1223  val_loss=2.1604  val_acc=0.1601  lr=1.56e-05
  ✅ Salvato best model (val_acc=0.1601)


  0%|          | 0/3235 [00:00<?, ?it/s]

  0%|          | 0/854 [00:00<?, ?it/s]

Epoch 02/20  train_loss=35.7628  train_acc=0.2786  val_loss=1.7147  val_acc=0.4218  lr=3.12e-05
  ✅ Salvato best model (val_acc=0.4218)


  0%|          | 0/3235 [00:00<?, ?it/s]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ep = range(1, len(history["train_loss"]) + 1)

axes[0].plot(ep, history["train_loss"], label="train")
axes[0].plot(ep, history["val_loss"],   label="val")
axes[0].set_title("Loss")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(ep, history["train_acc"], label="train")
axes[1].plot(ep, history["val_acc"],   label="val")
axes[1].set_title("Accuracy")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle("UNI + CropR Fine-tuning", fontsize=14)
plt.tight_layout()
plt.savefig(CFG["output_dir"] / "training_curves.png", dpi=150)
plt.show()

# %% [markdown]
# ## 9 · Valutazione sul Test Set

# %%
model.load_state_dict(
    torch.load(CFG["output_dir"] / "best_model.pt", map_location=device)
)
model.eval()

all_preds, all_labels = [], []

with torch.no_grad():
    for imgs, labels in tqdm(test_loader, desc="Test"):
        # In eval il modello restituisce un singolo tensore (inference=True in CropR)
        logits = model(imgs.to(device))
        preds  = logits.argmax(1).cpu()
        all_preds.append(preds)
        all_labels.append(labels)

all_preds  = torch.cat(all_preds).numpy()
all_labels = torch.cat(all_labels).numpy()

print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES))

# %% [markdown]
# ## 10 · Matrice di Confusione

# %%
cm = confusion_matrix(all_labels, all_preds, normalize="true")

fig, ax = plt.subplots(figsize=(8, 7))
sns.heatmap(
    cm, annot=True, fmt=".2f", cmap="Blues",
    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax,
)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title("Confusion Matrix (normalizzata) — UNI + CropR")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(CFG["output_dir"] / "confusion_matrix.png", dpi=150)
plt.show()

# %% [markdown]
# ## 11 · Token Budget (analisi CropR)
#
# Mostra quanti token rimangono dopo ogni blocco CropR, utile per verificare
# che la compressione sia coerente con la configurazione scelta.

# %%
if CFG["use_cropr"]:
    n_blk          = CFG["depth"]
    num_cropr_mods = n_blk - 2 if CFG["cropr_llf"] else n_blk - 1
    schedule       = [CFG["cropr_pruning_rate"]] * num_cropr_mods
    schedule[0]   += 1   # primo modulo rimuove un token in più

    num_tokens_initial = (CFG["img_size"] // CFG["patch_size"]) ** 2 + 1  # +1 CLS
    remaining = [
        num_tokens_initial - sum(schedule[: i + 1])
        for i in range(num_cropr_mods)
    ]

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(range(1, len(remaining) + 1), remaining, marker="o", linewidth=2)
    ax.axhline(num_tokens_initial, linestyle="--", color="gray", label=f"Token iniziali ({num_tokens_initial})")
    ax.set_xlabel("Blocco CropR")
    ax.set_ylabel("Token rimanenti")
    ax.set_title("Token Budget — progressione CropR")
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(CFG["output_dir"] / "token_budget.png", dpi=150)
    plt.show()

    compression = remaining[-1] / num_tokens_initial
    print(f"Token iniziali : {num_tokens_initial}")
    print(f"Token finali   : {remaining[-1]}")
    print(f"Compressione   : {compression:.1%}  ({1 - compression:.1%} rimossi)")

In [ ]:
report = classification_report(all_labels, all_preds, target_names=CLASS_NAMES, output_dict=True)
wandb.log({"test/" + k.replace(" ", "_"): v["f1-score"] 
           for k, v in report.items() if isinstance(v, dict)})
wandb.log({"test/accuracy": report["accuracy"]})

# Logga anche la confusion matrix come immagine
fig, ax = plt.subplots(figsize=(8, 7))
sns.heatmap(cm, annot=True, fmt=".2f", cmap="Blues",
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
wandb.log({"confusion_matrix": wandb.Image(fig)})
plt.close(fig)

wandb.finish()